# Lab Day 10 — Nguyễn Văn Thân
Triển khai QLoRA Qwen3-VL-4B trên Kaggle. Mã chi tiết ở `solution/`; cách điều khiển từ máy cá nhân ở `KAGGLE.md`. Notebook này để đọc và chạy từng phần; chiến dịch tự động dùng các notebook riêng cho mỗi setup. Các cell huấn luyện chỉ chạy trên Kaggle.


## 0. Môi trường (Kaggle T4)
Chỉ dùng một T4; GPU còn lại không được dùng. Chạy từ thư mục repo đã được đưa lên Kaggle.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HOME'] = '/tmp/lab-huggingface'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from pathlib import Path
assert Path('/kaggle/input').is_dir(), 'Chạy notebook này trên Kaggle.'
!nvidia-smi -L
!pip -q install -r requirements-colab.txt pytest datasets


In [ ]:
# Dataset công khai được gắn qua dataset_sources. Không cần đặt API key trong notebook.
import torch
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))


## 1. Dữ liệu: tải và chia split

In [ ]:
from scripts.prepare_data import _find_root, prepare
meta = prepare(_find_root('/kaggle/input'), 'data')
print(meta['dataset_version'], meta['counts'])


In [ ]:
from scripts.labcommon import read_jsonl
from solution.data import target_objects
from PIL import Image, ImageDraw
train = read_jsonl('data/splits/train.jsonl')
val = read_jsonl('data/splits/val.jsonl')
test = read_jsonl('data/splits/test.jsonl')
sessions = [{e['session'] for e in rows} for rows in (train, val, test)]
assert all(not sessions[i].intersection(sessions[j]) for i in range(3) for j in range(i))
for entry in train[:6]:
    image = Image.open(Path('data') / entry['image_path']).convert('RGB')
    drawing = ImageDraw.Draw(image)
    for box in target_objects(entry, 'pixel'):
        drawing.rectangle(box['bbox_2d'], outline='lime', width=2)
    display(image)


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from solution.data import answer, prompt, target_objects
entry = train[0]
print(prompt())
print(answer(entry))
print('Pixel:', target_objects(entry, 'pixel'))


## 3. E0: zero-shot

In [ ]:
# E0: chỉ chạy cell này khi cần thực hiện baseline lần đầu.
# Chiến dịch CLI tự chạy E0 trong kernel riêng và không chấm test lặp lại.
!python -u -m solution.run --phase e0


In [ ]:
import json
print(json.loads(Path('report/evidence/e0.json').read_text())['micro'])


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from solution.data import SupervisedDataset, collate
from solution.engine import diagnose
# diagnose(dataset, processor, model.config, output_path) kiểm tra 220 token ảnh
# và decode các labels != -100 thành JSON đáp án + token kết thúc.
# SupervisedDataset kiểm tra tiền tố và mask loss; collate pad labels=-100.
import inspect
print(inspect.getsource(SupervisedDataset))
print(inspect.getsource(collate))


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from solution.modeling import apply_setup, restore_checkpoint, save_checkpoint
print(inspect.getsource(apply_setup))
# E1/E2: 252 module; E3: 348 module. Projector trainable giữ float32.


## 6. Vòng huấn luyện

In [ ]:
from solution.engine import train_sft
print(inspect.getsource(train_sft))
# AdamW, warmup 5%, cosine, fp16 AMP, gradient checkpointing, clipping.
# Lưu riêng projector và chọn checkpoint theo validation; test không chọn model.


## 7. Suy luận và chấm E1–E3

In [ ]:
# Chọn một setup mỗi phiên GPU để giải phóng tài nguyên giữa các lượt chạy.
SETUP = 'e1'  # lần lượt e1, e2, e3
!python -u -m solution.run --phase {SETUP}


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4/E5 cần gắn output kernels E1, E2, E3 vào /kaggle/input.
# CLI tự gắn chúng, chọn baseline theo validation và giữ nguyên các biến còn lại.
!python -m pytest -q solution/test_solution.py
# Chạy từng phase trong kernel riêng:
# !python -u -m solution.run --phase e4
# !python -u -m solution.run --phase e5
# !python -u -m solution.run --phase report
